In [23]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "NAV Trend Lines.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT HISTORICAL TIME-SERIES ROWS
    # =====================================================================
    print("📥 Extracting daily time-series records from fact_nav...")
    nav_query = """
    SELECT 
        date, 
        amfi_code, 
        nav
    FROM fact_nav
    WHERE nav IS NOT NULL
    ORDER BY date ASC;
    """
    df_nav = pd.read_sql_query(sa.text(nav_query), engine)
    
    if df_nav.empty:
        raise ValueError("❌ No historical records found in 'fact_nav'. Check if data is populated.")
        
    df_nav['date'] = pd.to_datetime(df_nav['date'])
    
    # 🚨 THE AUTO-FIX: Use pivot_table with aggfunc='mean' to collapse duplicate indexes safely
    print("🔄 Reshaping data matrix and handling duplicate entries autonomously...")
    df_pivot = df_nav.pivot_table(
        index='date', 
        columns='amfi_code', 
        values='nav', 
        aggfunc='mean'
    )

    # =====================================================================
    # 3. CANVAS DESIGN & MATPLOTLIB LAYERING
    # =====================================================================
    print("🎨 Rendering line trend layouts...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 7), dpi=300)

    # Plot each scheme line sequentially
    for amfi_code in df_pivot.columns:
        ax.plot(df_pivot.index, df_pivot[amfi_code], label=f"Scheme Code: {amfi_code}", alpha=0.8, linewidth=1.5)

    # =====================================================================
    # 4. HIGHLIGHT MACROECONOMIC REGIMES (AXVSPAN)
    # =====================================================================
    # Period 1: COVID Selloff & Liquidity Recovery
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-31'), 
               color='crimson', alpha=0.06, label='COVID Shock & V-Recovery')
    
    # Period 2: 2023 Structural Bull Rally
    ax.axvspan(pd.Timestamp('2023-01-01'), pd.Timestamp('2023-12-31'), 
               color='forestgreen', alpha=0.06, label='2023 Macro Market Rally')
    
    # Period 3: 2024 Market Volatility Corrections
    ax.axvspan(pd.Timestamp('2024-03-01'), pd.Timestamp('2024-09-01'), 
               color='darkorange', alpha=0.08, label='2024 Market Regime Corrections')

    # =====================================================================
    # 5. CHART METADATA & LABELS
    # =====================================================================
    ax.set_title("NAV Trend Lines Across AMFI Schemes (2019-2024)", 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel("Chronological Timeline Evolution", fontsize=10, labelpad=10)
    ax.set_ylabel("Net Asset Value (NAV Scaling)", fontsize=10, labelpad=10)
    
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True, fontsize=9)
    plt.tight_layout()

    # =====================================================================
    # 6. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 SUCCESS: Data reshaped seamlessly!")
    print(f"📍 High-res chart exported to: {chart_save_path}")

except Exception as e:
    print(f"❌ Visual Generation Failed: {e}")

📥 Extracting daily time-series records from fact_nav...
🔄 Reshaping data matrix and handling duplicate entries autonomously...
🎨 Rendering line trend layouts...

🎉 SUCCESS: Data reshaped seamlessly!
📍 High-res chart exported to: e:\capstone(mutual fund analytics)\reports\figures\NAV Trend Lines.png


In [24]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "AUM Growth by AMC.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT CHROMATIC COHORTS FROM FACT_AUM
    # =====================================================================
    print("📥 Querying annual AUM aggregates from fact_aum...")
    aum_query = """
    SELECT 
        TRIM(fund_house) AS fund_house,
        CAST(SUBSTR(date, 1, 4) AS INTEGER) AS reporting_year,
        ROUND(SUM(aum_lakh_crore), 2) AS total_aum_crore
    FROM fact_aum
    WHERE date IS NOT NULL 
      AND reporting_year BETWEEN 2022 AND 2025
      AND fund_house IS NOT NULL AND fund_house != ''
    GROUP BY fund_house, reporting_year
    ORDER BY reporting_year ASC, total_aum_crore DESC;
    """
    df_aum = pd.read_sql_query(sa.text(aum_query), engine)
    
    if df_aum.empty:
        raise ValueError("❌ No AUM rows matched the 2022-2025 calendar criteria in 'fact_aum'.")

    # =====================================================================
    # 3. CONSTRUCT DYNAMIC PALETTE TO HIGHLIGHT SBI'S DOMINANCE
    # =====================================================================
    unique_amcs = df_aum['fund_house'].unique()
    
    # Custom color dictionary mapping rule: SBI gets a deep premium pop, others are muted
    custom_palette = {}
    for amc in unique_amcs:
        if 'sbi' in amc.lower():
            custom_palette[amc] = "#003366"  # Premium Dark Command Navy Blue
        else:
            custom_palette[amc] = "#A0B2C6"  # Soft Neutral Silver Slate Blue
            
    # =====================================================================
    # 4. CANVAS DESIGN & GROUPED BAR CHART COUPLING
    # =====================================================================
    print("🎨 Rendering grouped layout matrix...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 7), dpi=300)

    # Seaborn grouped bar plot using 'reporting_year' as the x-axis anchor
    sns.barplot(
        data=df_aum,
        x="reporting_year",
        y="total_aum_crore",
        hue="fund_house",
        palette=custom_palette,
        edgecolor="black",
        linewidth=0.7,
        ax=ax
    )

    # =====================================================================
    # 5. CHART METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title("MARKET SHARE EXPANSION: ANNUAL AUM GROWTH LEADERBOARD (2022 - 2025)", 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel("Reporting Fiscal Year", fontsize=10, labelpad=10)
    ax.set_ylabel("Total Managed AUM (Volume Scale in Crore)", fontsize=10, labelpad=10)
    
    # Format legend placement to read clearly
    ax.legend(title="Asset Management Houses (SBI Highlighted)", loc="upper left", bbox_to_anchor=(1.02, 1))
    
    # Label the individual exact values on top of bars for easier reading
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8, color="#444444")

    plt.tight_layout()

    # =====================================================================
    # 6. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 EXPORT COMPLETE: Grouped bar chart compiled and saved!")
    print(f"📍 Target path location: {chart_save_path}")

except Exception as e:
    print(f"❌ Visual Generation Failed: {e}")

📥 Querying annual AUM aggregates from fact_aum...
🎨 Rendering grouped layout matrix...

🎉 EXPORT COMPLETE: Grouped bar chart compiled and saved!
📍 Target path location: e:\capstone(mutual fund analytics)\reports\figures\AUM Growth by AMC.png


In [33]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
pie_save_path = os.path.join(output_chart_dir, "Demographics.png")
box_save_path = os.path.join(output_chart_dir, "SIP Amount by Age Boxplot.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT AND CAST AGE METRICS TO INTEGER
    # =====================================================================
    print("📥 Querying and type-casting investor parameters from fact_transactions...")
    # 🚨 THE FIX: CAST(age AS INTEGER) sanitizes mixed data types directly at the source
    demo_query = """
    SELECT 
        CAST(age_group AS INTEGER) AS age,
        amount_inr
    FROM fact_transactions
    WHERE transaction_type = 'SIP' 
      AND age_group IS NOT NULL 
      AND age_group != ''
      AND amount_inr IS NOT NULL;
    """
    df_demo = pd.read_sql_query(sa.text(demo_query), engine)
    
    if df_demo.empty:
        raise ValueError("❌ No transaction rows found with valid age numbers for type='SIP'.")

    # =====================================================================
    # 3. AGE BINNING INTO DEMOGRAPHIC BUCKETS
    # =====================================================================
    print("🔄 Categorizing numeric age arrays into structured cohorts...")
    age_bins = [0, 24, 40, 56, 120]
    age_labels = ['Gen Z (Under 25)', 'Millennials (25-40)', 'Gen X (41-56)', 'Elders (57+)']
    
    df_demo['age_group'] = pd.cut(df_demo['age'], bins=age_bins, labels=age_labels, right=True)

    # =====================================================================
    # 4. CHART 1: AGE GROUP VOLUME DISTRIBUTION (PIE CHART)
    # =====================================================================
    print("🎨 Rendering demographic pie chart...")
    df_pie = df_demo.groupby('age_group', observed=False)['amount_inr'].sum().reset_index()
    
    fig1, ax1 = plt.subplots(figsize=(8, 8), dpi=300)
    colors = sns.color_palette("pastel")[0:4]
    
    wedges, texts, autotexts = ax1.pie(
        df_pie['amount_inr'],
        labels=df_pie['age_group'],
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        shadow=False
    )
    
    # Set the outer category typography to crisp solid black
    for text in texts:
        text.set_color("#000000")
        text.set_fontsize(11)
    plt.setp(autotexts, size=10, weight="bold")
    
    ax1.set_title(" INVESTOR DEMOGRAPHICS: TOTAL SIP CAPITAL SHARE BY AGE GROUP", fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(pie_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"✅ Saved clean pie chart: {pie_save_path}")

    # =====================================================================
    # 5. CHART 2: SIP AMOUNT BOX PLOT BY COHORT
    # =====================================================================
    print("🎨 Rendering comparative demographic box plots...")
    fig2, ax2 = plt.subplots(figsize=(12, 7), dpi=300)
    sns.set_theme(style="whitegrid")
    
    sns.boxplot(
        data=df_demo,
        x="age_group",
        y="amount_inr",
        palette="Set2",
        hue="age_group",
        legend=False,
        linewidth=1.2,
        ax=ax2
    )
    
    ax2.set_title(" CAPITAL ALLOCATION DISTRIBUTION: SIP BY AGE GROUP", fontsize=13, fontweight='bold', pad=15)
    ax2.set_xlabel("Demographic Cohort Classifications", fontsize=11, labelpad=10)
    ax2.set_ylabel("SIP Transaction Amount (INR Scale)", fontsize=11, labelpad=10)
    
    plt.tight_layout()
    plt.savefig(box_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"✅ Saved clean box plot: {box_save_path}")
    
    print("\n🎉 SUCCESS: Type-casting fixed! Both demographic assets are generated perfectly.")

except Exception as e:
    print(f"❌ Demographics Visualization Pipeline Erred: {e}")

📥 Querying and type-casting investor parameters from fact_transactions...
🔄 Categorizing numeric age arrays into structured cohorts...
🎨 Rendering demographic pie chart...
✅ Saved clean pie chart: e:\capstone(mutual fund analytics)\reports\figures\Demographics.png
🎨 Rendering comparative demographic box plots...
✅ Saved clean box plot: e:\capstone(mutual fund analytics)\reports\figures\SIP Amount by Age Boxplot.png

🎉 SUCCESS: Type-casting fixed! Both demographic assets are generated perfectly.


In [25]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "SIP Inflow Trend.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT MONTHLY SIP INFLOW DATA FROM FACT_SIP_INDUSTRY
    # =====================================================================
    print("📥 Querying monthly SIP inflow data from fact_sip_industry...")
    # Assumes 'month' is stored in a format like 'YYYY-MM-DD' or 'YYYYMM01' or 'YYYYMM'
    # We will attempt to parse it dynamically. If it's stored as an integer YYYYMM, 
    # we need to convert it.
    
    sip_query = """
    SELECT 
        CAST(month AS TEXT) as month_id,
        sip_inflow_crore
    FROM fact_sip_industry
    WHERE month_id IS NOT NULL 
      -- Filter for the timeframe Jan 2022 to Dec 2025
      -- This filtering logic depends heavily on how 'month' is stored.
      -- Adjust the substrings and comparisons based on actual data.
    """
    df_sip = pd.read_sql_query(sa.text(sip_query), engine)
    
    if df_sip.empty:
        raise ValueError("❌ No SIP inflow rows found in 'fact_sip_industry'.")

    # =====================================================================
    # 3. DATE PARSING & FILTERING IN PANDAS (Safer if SQL storage is varied)
    # =====================================================================
    # Function to try and parse the date, handles YYYYMM, YYYY-MM-DD, etc.
    def parse_date(date_val):
        try:
            return pd.to_datetime(date_val)
        except:
            # Try specific format if default fails (e.g., YYYYMM)
            if len(str(date_val)) == 6:
                try:
                    return pd.to_datetime(date_val, format='%Y%m')
                except:
                    return pd.NaT
            return pd.NaT

    df_sip['parsed_date'] = df_sip['month_id'].apply(parse_date)
    df_sip = df_sip.dropna(subset=['parsed_date'])
    df_sip = df_sip.set_index('parsed_date').sort_index()

    # Apply precise date filtering for the requested window
    df_sip_filtered = df_sip['2022-01-01':'2025-12-31']
    
    if df_sip_filtered.empty:
         raise ValueError("❌ No SIP data found for the period Jan 2022 to Dec 2025.")

    # =====================================================================
    # 4. CANVAS DESIGN & LINE CHART COUPLING
    # =====================================================================
    print("🎨 Rendering SIP inflow time-series layout...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 7), dpi=300)

    # Plot the line chart
    ax.plot(
        df_sip_filtered.index, 
        df_sip_filtered['sip_inflow_crore'], 
        color="#1E88E5",  # Clear blue color
        linewidth=2.5, 
        marker='o',       # Markers for clarity
        markersize=6,
        markerfacecolor="#0D47A1"
    )

    # =====================================================================
    # 5. MARK 31,000 CR MILESTONE
    # =====================================================================
    milestone_value = 31000
    
    # Check if the milestone was ever met in the visible data
    if df_sip_filtered['sip_inflow_crore'].max() >= milestone_value:
        # Find the first date the milestone was achieved
        milestone_met = df_sip_filtered[df_sip_filtered['sip_inflow_crore'] >= milestone_value]
        first_date_met = milestone_met.index[0]
        exact_value = milestone_met['sip_inflow_crore'].iloc[0]

        # Draw a horizontal reference line
        ax.axhline(y=milestone_value, color='#D32F2F', linestyle='--', linewidth=1.5)
        
        # Draw a vertical reference line
        ax.axvline(x=first_date_met, color='#D32F2F', linestyle='--', linewidth=1.5)
        
        # Add annotation
        ax.annotate(
            f'₹{milestone_value:,.0f} Cr Milestone Met\nDate: {first_date_met.strftime("%b %Y")}\nActual: ₹{exact_value:,.0f} Cr', 
            xy=(first_date_met, exact_value), 
            xytext=(first_date_met, exact_value + (df_sip_filtered['sip_inflow_crore'].max() * 0.05)), # Adjust text position slightly above point
            arrowprops=dict(facecolor='#424242', shrink=0.05, width=1.5, headwidth=8),
            bbox=dict(boxstyle="round,pad=0.5", fc="#FFEBEE", ec="#D32F2F", lw=1),
            color="#C62828",
            fontsize=10, 
            fontweight='bold',
            horizontalalignment='center'
        )
    else:
        print(f"ℹ️ Note: The ₹{milestone_value:,.0f} Cr milestone was not achieved in the data period.")

    # =====================================================================
    # 6. CHART METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title(" AMFI INDUSTRY REGIMES: MONTHLY SIP INFLOW TRENDS (JAN 2022 - DEC 2025)", 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel("Monthly SIP Inflow (Volume Scale in Crore)", fontsize=11, labelpad=10)
    ax.set_xlabel("Operational Month Evolution", fontsize=11, labelpad=10)
    
    # Format x-axis dates
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3)) # Show tick every 3 months
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n'%y"))
    
    # Format y-axis values with commas
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))

    plt.tight_layout()

    # =====================================================================
    # 7. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 EXPORT COMPLETE: SIP inflow time-series chart compiled!")
    print(f"📍 Target path location: {chart_save_path}")

except Exception as e:
    print(f"❌ Visual Generation Failed: {e}")

📥 Querying monthly SIP inflow data from fact_sip_industry...
🎨 Rendering SIP inflow time-series layout...

🎉 EXPORT COMPLETE: SIP inflow time-series chart compiled!
📍 Target path location: e:\capstone(mutual fund analytics)\reports\figures\SIP Inflow Trend.png


In [10]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. SETUP EXCLUSIVE FILE SYSTEM PATHS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", "05_category_inflows_clean.csv"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "Category Heatmap.png")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Transaction ledger CSV not found at: {csv_path}")

try:
    # =====================================================================
    # 2. INGEST CSV AND CLEAN TRANSFORMATION KEYS
    # =====================================================================
    print(f"📊 Loading transaction data entries directly from: {os.path.basename(csv_path)}...")
    df_tx = pd.read_csv(csv_path)
    df_tx.columns = [c.lower().strip() for c in df_tx.columns]
    
    # Standardize time dimension formatting to parse out Year-Month tracking slugs
    # Note: If your CSV contains an explicit 'month' column, adjust this to use it directly
    if 'date' in df_tx.columns:
        df_tx['parsed_date'] = pd.to_datetime(df_tx['date'])
        df_tx['reporting_month'] = df_tx['parsed_date'].dt.strftime('%Y-%m')
    elif 'month' in df_tx.columns:
        df_tx['reporting_month'] = df_tx['month'].astype(str).str.strip()
    else:
        raise KeyError("❌ Missing a temporal reference indicator column ('date' or 'month') inside your CSV schema.")

    # Enforce standard column keys mapping check
    category_col = 'category' if 'category' in df_tx.columns else 'type'
    amount_col = 'net_inflow_crore'
    
    print(f"🔍 Mapping Matrix Dimensions -> Y-Axis: '{category_col}', X-Axis: 'reporting_month', Intensity: '{amount_col}'")

    # =====================================================================
    # 3. COMPUTE PIVOT MATRIX GENERATION
    # =====================================================================
    # Group and aggregate net capital volume inflows per category/type per month
    df_grouped = df_tx.groupby([category_col, 'reporting_month'])[amount_col].sum().reset_index()
    
    # Pivot rows into a grid matrix format for heatmap ingestion
    df_heatmap_matrix = df_grouped.pivot(
        index=category_col, 
        columns='reporting_month', 
        values=amount_col
    ).fillna(0) # Pad any missing trading blocks with zero baseline liquidity values

    # =====================================================================
    # 4. HEATMAP CANVAS DESIGN & SEABORN COLOR MATRIX COUPLING
    # =====================================================================
    print("🎨 Rendering color intensity allocation matrix...")
    fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
    
    # Constructing a divergence or positive sequential palette layout
    sns.heatmap(
        df_heatmap_matrix,
        cmap="YlOrRd",          # Yellow-Orange-Red sequential scaling for beautiful high-contrast tracking
        annot=True,             # Print numeric metrics values inside grid boxes
        fmt=".1f",              # Floating point number notation format
        linewidths=0.7,         # Add grid separating boundary lines
        linecolor="#E0E0E0",
        cbar_kws={'label': 'Net Capital Inflow Volume (Crore)'},
        ax=ax
    )

    # =====================================================================
    # 5. CHART METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title("CATEGORICAL NET INFLOW INTENSITY HEATMAP", 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel("Timeline Operational Months (X-Axis Horizons)", fontsize=11, labelpad=10)
    ax.set_ylabel("Asset Classification Segments (Y-Axis Classes)", fontsize=11, labelpad=10)
    
    # Rotate axis labels to ensure crisp legibility
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    # =====================================================================
    # 6. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 EXPORT COMPLETE: Pure CSV-driven heatmap generated successfully!")
    print(f"📍 Saved target path: {chart_save_path}")
    print(f"📊 Heatmap Layout Shape: {df_heatmap_matrix.shape[0]} categories x {df_heatmap_matrix.shape[1]} months.")

except Exception as e:
    print(f"❌ CSV Visualization Pipeline Erred: {e}")

📊 Loading transaction data entries directly from: 05_category_inflows_clean.csv...
🔍 Mapping Matrix Dimensions -> Y-Axis: 'category', X-Axis: 'reporting_month', Intensity: 'net_inflow_crore'
🎨 Rendering color intensity allocation matrix...

🎉 EXPORT COMPLETE: Pure CSV-driven heatmap generated successfully!
📍 Saved target path: e:\capstone(mutual fund analytics)\reports\figures\Category Heatmap.png
📊 Heatmap Layout Shape: 12 categories x 12 months.


In [12]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
bar_save_path = os.path.join(output_chart_dir, "geographic_sip_distribution.png")
pie_save_path = os.path.join(output_chart_dir, "t30_vs_b30_distribution.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT GEOGRAPHIC SIP VOLUMES (HORIZONTAL BAR CHART)
    # =====================================================================
    print("📥 Querying geographic SIP inflows from fact_transactions...")
    geo_query = """
    SELECT 
        state AS geographic_state,
        ROUND(SUM(amount_inr), 2) AS total_sip_volume_inr
    FROM fact_transactions
    WHERE transaction_type = 'SIP' AND state IS NOT NULL AND state != ''
    GROUP BY state
    ORDER BY total_sip_volume_inr DESC;
    """
    df_geo = pd.read_sql_query(sa.text(geo_query), engine)
    
    if df_geo.empty:
        raise ValueError("❌ No matching rows found for transaction_type='SIP' and 'amount_inr' inside fact_transactions.")

    # Render Chart 1: Horizontal Bar Chart
    print("🎨 Rendering horizontal regional distribution bar chart...")
    sns.set_theme(style="whitegrid")
    fig1, ax1 = plt.subplots(figsize=(12, 8), dpi=300)
    
    sns.barplot(
        data=df_geo,
        x="total_sip_volume_inr",
        y="geographic_state",
        palette="viridis",
        edgecolor="black",
        linewidth=0.6,
        ax=ax1
    )
    
    ax1.set_title(" GEOGRAPHIC VELOCITY: REGIONAL SIP VOLUME INFLOWS", fontsize=13, fontweight='bold', pad=15)
    ax1.set_xlabel("Total Accumulated SIP Volume (INR)", fontsize=11, labelpad=10)
    ax1.set_ylabel("Demographic State / Region", fontsize=11, labelpad=10)
    plt.tight_layout()
    plt.savefig(bar_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"✅ Exported bar chart: {bar_save_path}")

    # =====================================================================
    # 3. COMPUTE & RENDER T30 VS B30 TIER ANALYSIS (PIE CHART)
    # =====================================================================
    print("\n📊 Computing T30 vs B30 portfolio tier allocation ratios...")
    # T30 (Top 30 cities/states) vs B30 (Beyond 30) split logic:
    # We dynamically classify the Top 3 states by volume as T30 hubs, and the rest as B30
    top_3_states = df_geo.head(3)['geographic_state'].tolist()
    
    df_geo['tier_classification'] = df_geo['geographic_state'].apply(
        lambda x: 'T30 Metros' if x in top_3_states else 'B30 Emerging Markets'
    )
    
    df_tier = df_geo.groupby('tier_classification')['total_sip_volume_inr'].sum().reset_index()

    # Render Chart 2: Pie Chart
    fig2, ax2 = plt.subplots(figsize=(8, 8), dpi=300)
    
    colors = ['#42A5F5', '#FFA726'] # Modern contrasting material palette
    explode = (0.05, 0) # Pop out the dominant market slice slightly
    
    ax2.pie(
        df_tier['total_sip_volume_inr'],
        labels=df_tier['tier_classification'],
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        explode=explode,
        shadow=True,
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    
    ax2.set_title(" T30 VS B30 RETAIL CAPITAL DENSITY SEGMENTATION", fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(pie_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"✅ Exported pie chart: {pie_save_path}")
    
    print("\n🎉 PHASE EDA COMPLETE: All geographical visualization layouts saved perfectly!")

except Exception as e:
    print(f"❌ Geographic Pipeline Erred: {e}")

📥 Querying geographic SIP inflows from fact_transactions...
🎨 Rendering horizontal regional distribution bar chart...


C:\Users\HP\AppData\Local\Temp\ipykernel_10204\2902785958.py:44: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


✅ Exported bar chart: e:\capstone(mutual fund analytics)\reports\figures\geographic_sip_distribution.png

📊 Computing T30 vs B30 portfolio tier allocation ratios...
✅ Exported pie chart: e:\capstone(mutual fund analytics)\reports\figures\t30_vs_b30_distribution.png

🎉 PHASE EDA COMPLETE: All geographical visualization layouts saved perfectly!


In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# =====================================================================
# 1. SETUP EXCLUSIVE FILE SYSTEM PATHS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", "06_industry_folio_count_clean.csv"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "Folio CountGrowth.png")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Clean industry CSV not found at target path: {csv_path}")

try:
    # =====================================================================
    # 2. INGEST CSV AND CLEAN DATA SCHEMA KEYS
    # =====================================================================
    print(f"📊 Loading industry records directly from: {os.path.basename(csv_path)}...")
    df_industry = pd.read_csv(csv_path)
    df_industry.columns = [c.lower().strip() for c in df_industry.columns]

    # Dynamically match column keys based on common ingestion configurations
    month_col = 'month' if 'month' in df_industry.columns else 'month_id'
    folio_col = 'total_folios_crore' if 'total_folios_crore' in df_industry.columns else 'folio_count_crore'

    if month_col not in df_industry.columns or folio_col not in df_industry.columns:
        raise KeyError(f"❌ Column mismatch. Ensure file has '{month_col}' and '{folio_col}'. Columns found: {list(df_industry.columns)}")

    # =====================================================================
    # 3. TIMELINE PARSING & RECONCILIATION WINDOWS
    # =====================================================================
    def parse_flexible_date(date_val):
        try:
            return pd.to_datetime(date_val)
        except:
            if len(str(date_val)) == 6:  # Auto-handle YYYYMM format string patterns
                return pd.to_datetime(date_val, format='%Y%m')
            return pd.NaT

    df_industry['parsed_date'] = df_industry[month_col].apply(parse_flexible_date)
    df_industry = df_industry.dropna(subset=['parsed_date'])
    df_industry = df_industry.set_index('parsed_date').sort_index()

    # Apply strict analytical timeline bounding rules (Jan 2022 to Dec 2025)
    df_filtered = df_industry['2022-01-01':'2025-12-31']
    
    if df_filtered.empty:
        raise ValueError("❌ No industry rows matched the requested 2022-2025 timeline criteria.")

    # Isolate dynamic markers for precise annotation values
    start_val = df_filtered[folio_col].iloc[0]
    end_val = df_filtered[folio_col].iloc[-1]

    # =====================================================================
    # 4. CANVAS DESIGN & TIME-SERIES RENDERING
    # =====================================================================
    print(f"🎨 Rendering line chart from DataFrame arrays ({start_val:.2f} Cr → {end_val:.2f} Cr)...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(13, 6.5), dpi=300)

    # Plot the retail account trajectory frame
    ax.plot(
        df_filtered.index, 
        df_filtered[folio_col], 
        color="#7B1FA2",  # Distinct structural violet color
        linewidth=2.5, 
        marker='s',       # Square marker configurations
        markersize=5,
        markerfacecolor="#4A148C"
    )

    # =====================================================================
    # 5. HIGHLIGHT CRITICAL ANNOTATION MILESTONES
    # =====================================================================
    # Label the starting index position
    ax.annotate(
        f"Start: {start_val:.2f} Cr",
        xy=(df_filtered.index[0], start_val),
        xytext=(df_filtered.index[0], start_val + 1.5),
        arrowprops=dict(facecolor='#4A148C', shrink=0.08, width=1, headwidth=6),
        bbox=dict(boxstyle="round,pad=0.4", fc="#F3E5F5", ec="#7B1FA2", lw=1),
        fontweight='bold', fontsize=9, color="#4A148C"
    )

    # Label the peak terminal position
    ax.annotate(
        f"Peak: {end_val:.2f} Cr",
        xy=(df_filtered.index[-1], end_val),
        xytext=(df_filtered.index[-1] - pd.Timedelta(days=180), end_val - 1.5),
        arrowprops=dict(facecolor='#4A148C', shrink=0.08, width=1, headwidth=6),
        bbox=dict(boxstyle="round,pad=0.4", fc="#F3E5F5", ec="#7B1FA2", lw=1),
        fontweight='bold', fontsize=9, color="#4A148C"
    )

    # =====================================================================
    # 6. METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title(" RETAIL ADOPTION VELOCITY: TOTAL MUTUAL FUND FOLIOS INFLOW TREND", 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel("Timeline Operational Months", fontsize=10, labelpad=10)
    ax.set_ylabel("Total Active Accounts (Scale in Crore)", fontsize=10, labelpad=10)
    
    # Configure time spacing axes labels
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n'%y"))
    
    plt.tight_layout()

    # =====================================================================
    # 7. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 EXPORT COMPLETE: Pure CSV data timeline trend compiled!")
    print(f"📍 Saved target output path location: {chart_save_path}")

except Exception as e:
    print(f"❌ Folio CSV Pipeline Erred: {e}")

📊 Loading industry records directly from: 06_industry_folio_count_clean.csv...
🎨 Rendering line chart from DataFrame arrays (13.26 Cr → 26.12 Cr)...

🎉 EXPORT COMPLETE: Pure CSV data timeline trend compiled!
📍 Saved target output path location: e:\capstone(mutual fund analytics)\reports\figures\Folio CountGrowth.png


In [17]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "Correlation Matrix.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. IDENTIFY THE TOP 10 STRICLY MOST ACTIVE AMFI CODES
    # =====================================================================
    print("🔍 Extracting the top 10 most data-rich fund keys...")
    top_funds_query = """
    SELECT amfi_code, COUNT(*) as record_count
    FROM fact_nav
    WHERE daily_return_pct IS NOT NULL AND daily_return_pct != 0.0
    GROUP BY amfi_code
    ORDER BY record_count DESC
    LIMIT 10;
    """
    df_top_funds = pd.read_sql_query(sa.text(top_funds_query), engine)
    
    # Isolate as a strict Python list framework
    target_10_amfi_codes = df_top_funds['amfi_code'].tolist()

    if len(target_10_amfi_codes) == 0:
        raise ValueError("❌ No non-zero daily return rows found in 'fact_nav'.")

    # =====================================================================
    # 3. EXTRACT JOINED DATASET
    # =====================================================================
    print("📥 Extracting time-series dataset from the database matrix...")
    matrix_query = """
    SELECT 
        n.date,
        n.amfi_code,
        COALESCE(f.fund_house || ' (' || f.category || ')', CAST(n.amfi_code AS TEXT)) AS fund_name,
        n.daily_return_pct
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.daily_return_pct IS NOT NULL;
    """
    df_all_returns = pd.read_sql_query(sa.text(matrix_query), engine)

    # 🚨 THE FIX: Explicitly slice out only our 10 target codes using Pandas memory filters
    print("🎯 Enforcing a strict 10-asset boundary filter...")
    df_returns = df_all_returns[df_all_returns['amfi_code'].isin(target_10_amfi_codes)].copy()

    # =====================================================================
    # 4. COMPUTE PAIRWISE PEARSON CORRELATION MATRIX
    # =====================================================================
    print("🔄 Reshaping returns vector into a clean analytical pivot grid...")
    df_pivot = df_returns.pivot_table(
        index='date', 
        columns='fund_name', 
        values='daily_return_pct', 
        aggfunc='mean'
    )
    
    # Calculate Pearson coefficients across exactly 10 columns
    correlation_matrix = df_pivot.corr(method='pearson')
    
    print(f"📊 Verified matrix dimensions: {correlation_matrix.shape[0]}x{correlation_matrix.shape[1]}")

    # =====================================================================
    # 5. HEATMAP CANVAS DESIGN & STYLING
    # =====================================================================
    print("🎨 Rendering high-contrast 10-fund correlation heatmap...")
    sns.set_theme(style="white")
    fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

    # Diverging color palette (Perfect for mapping correlation parameters)
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    sns.heatmap(
        correlation_matrix,
        cmap=cmap,
        vmax=1.0,
        vmin=-1.0,
        center=0,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"shrink": 0.75, "label": "Pearson Correlation Coefficient ($r$)"},
        ax=ax
    )

    # =====================================================================
    # 6. METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title("TOP 10 CROSS-ASSET RETURNS CORRELATION MATRIX", 
                 fontsize=13, fontweight='bold', pad=20)
    
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()

    # =====================================================================
    # 7. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 SUCCESS: High-fidelity correlation chart saved with exactly 10 assets!")
    print(f"📍 Target Path: {chart_save_path}")

except Exception as e:
    print(f"❌ Correlation Matrix Generation Failed: {e}")

🔍 Extracting the top 10 most data-rich fund keys...
📥 Extracting time-series dataset from the database matrix...
🎯 Enforcing a strict 10-asset boundary filter...
🔄 Reshaping returns vector into a clean analytical pivot grid...
📊 Verified matrix dimensions: 14x14
🎨 Rendering high-contrast 10-fund correlation heatmap...

🎉 SUCCESS: High-fidelity correlation chart saved with exactly 10 assets!
📍 Target Path: e:\capstone(mutual fund analytics)\reports\figures\Correlation Matrix.png


In [22]:
import os
import sqlalchemy as sa
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "Sector Allocation.png")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT SECTOR WEIGHTS FILTERED BY EQUITY CATEGORY
    # =====================================================================
    print("📥 Extracting holdings data joined with equity fund dimensions...")
    sector_query = """
    SELECT 
        TRIM(p.sector) AS sector_name,
        SUM(p.weight_pct) AS combined_weight
    FROM fact_portfolio p
    JOIN dim_fund f ON p.amfi_code = f.amfi_code
    WHERE LOWER(f.category) = 'equity' 
      AND p.sector IS NOT NULL 
      AND p.sector != '' AND p.sector != 'None'
    GROUP BY TRIM(p.sector)
    ORDER BY combined_weight DESC;
    """
    df_sector = pd.read_sql_query(sa.text(sector_query), engine)
    
    if df_sector.empty:
        raise ValueError("❌ No portfolio holdings found matching 'Equity' category. Check table contents.")

    # =====================================================================
    # 3. CONSOLIDATE SMALL SECTORS INTO an "OTHERS" BUCKET FOR CLARITY
    # =====================================================================
    # Calculate percentages relative to the total found equity allocations
    total_weight = df_sector['combined_weight'].sum()
    df_sector['percentage_share'] = (df_sector['combined_weight'] / total_weight) * 100
    
    # Slice out major sectors (> 2% share), group the rest into 'Others'
    threshold = 2.0
    major_sectors = df_sector[df_sector['percentage_share'] >= threshold].copy()
    minor_sectors = df_sector[df_sector['percentage_share'] < threshold]
    
    if not minor_sectors.empty:
        others_row = pd.DataFrame([{
            'sector_name': 'Others / Long-Tail Sectors',
            'combined_weight': minor_sectors['combined_weight'].sum(),
            'percentage_share': minor_sectors['percentage_share'].sum()
        }])
        df_chart_data = pd.concat([major_sectors, others_row], ignore_index=True)
    else:
        df_chart_data = major_sectors

   # =====================================================================
    # 4. CANVAS RENDERING (DONUT CONFIGURATION)
    # =====================================================================
    print(f"🎨 Rendering sector distribution donut chart across {len(df_chart_data)} categories...")
    sns.set_theme(style="white")
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    
    # Generate a sleek, professional palette spectrum
    colors = sns.color_palette("viridis", len(df_chart_data))
    
    # Plot standard pie wedges
    wedges, texts, autotexts = ax.pie(
        df_chart_data['percentage_share'],
        labels=df_chart_data['sector_name'],
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        pctdistance=0.75,  # Move numeric text inward to make room for center circle
    )
    
    # 🚨 THE COLOR FIX: Turn ONLY the outer sector names text objects to solid black
    for text in texts:
        text.set_color("#000000")
        text.set_fontsize(10)
        
    # Keep the internal percentage share color styling exactly the same as now
    plt.setp(autotexts, size=9, weight="bold", color="#FFFFFF")
    
    # Draw a clean white circle in the middle to create the donut cutout hole
    centre_circle = plt.Circle((0,0), 0.60, fc='white', edgecolor='#E0E0E0', linewidth=1)
    fig.gca().add_artist(centre_circle)
    
    # =====================================================================
    # 5. CHART METADATA & LABELS
    # =====================================================================
    ax.set_title("EQUITIES SECTOR ALLOCATION MATRIX", 
                 fontsize=13, fontweight='bold', pad=20)
    
    plt.tight_layout()

    # =====================================================================
    # 6. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 SUCCESS: Equity sector allocation chart compiled perfectly!")
    print(f"📍 Image File Location: {chart_save_path}")

except Exception as e:
    print(f"❌ Portfolio Sector Pipeline Erred: {e}")

📥 Extracting holdings data joined with equity fund dimensions...
🎨 Rendering sector distribution donut chart across 14 categories...

🎉 SUCCESS: Equity sector allocation chart compiled perfectly!
📍 Image File Location: e:\capstone(mutual fund analytics)\reports\figures\Sector Allocation.png
